In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as scp
from scipy.optimize import brentq
import Black_Scholes as BS

## Loading the NSE option chain

The raw NSE CSV has a **two-line header** — a `CALLS , , PUTS` banner, then the real column names — and the labels `LTP`, `IV`, `OI`, `VOLUME` each appear **twice**: once for the call side (left of `STRIKE`) and once for the put side (right of it). That is why `df['CALLS']['LTP']` fails — after a plain `read_csv` there is no clean `LTP` column.

So we skip the two header lines, assign our own positional column names, and strip the thousands-commas and the `-` placeholders before converting everything to numbers.

In [ ]:
# our own names for the 23 columns (a blank one sits at each end of every NSE row)
cols = ['blank_l',
        'C_OI','C_CHNG_OI','C_VOLUME','C_IV','C_LTP','C_CHNG','C_BIDQTY','C_BID','C_ASK','C_ASKQTY',
        'STRIKE',
        'P_BIDQTY','P_BID','P_ASK','P_ASKQTY','P_CHNG','P_LTP','P_IV','P_VOLUME','P_CHNG_OI','P_OI',
        'blank_r']

df = pd.read_csv('option-chain-ED-NIFTY-25-Aug-2026.csv', skiprows=2, header=None, names=cols)
df = df.drop(columns=['blank_l', 'blank_r'])

def to_num(s):
    # '3,100.00' -> 3100.0 ,  '-' -> NaN
    return pd.to_numeric(
        s.astype(str).str.replace(',', '', regex=False).str.strip().replace('-', np.nan),
        errors='coerce')

optionChain = df.apply(to_num)
optionChain.head()

## Separating strike and LTP

`STRIKE` is the shared middle column; the last traded price lives in `C_LTP` for calls and `P_LTP` for puts. We pull each side into its own tidy frame (dropping strikes that never traded, i.e. `LTP` is blank), using `MarketPrice` / `StrikePrice` so the rest of the notebook can price against them.

In [ ]:
calls = (pd.DataFrame({'StrikePrice': optionChain['STRIKE'], 'MarketPrice': (optionChain['C_BID'] + optionChain['C_ASK']) / 2, 'IV': optionChain['C_IV']})
           .dropna(subset=['MarketPrice']).reset_index(drop=True))
puts  = (pd.DataFrame({'StrikePrice': optionChain['STRIKE'], 'MarketPrice': (optionChain['P_BID'] + optionChain['P_ASK']) / 2, 'IV': optionChain['P_IV']})
           .dropna(subset=['MarketPrice']).reset_index(drop=True))

print(f'{len(calls)} call strikes, {len(puts)} put strikes')
calls.head()

## The volatility smile

With strike and IV separated we can plot the smile directly from the exchange's IV column. Deep in-the-money options are illiquid and their quoted IVs are unreliable, so the clean way to draw the smile is to use **out-of-the-money** options on each side: puts below spot, calls above it.

> The spot below is inferred from put-call parity around the at-the-money strike (where call LTP ≈ put LTP). Replace it with the actual underlying level from your snapshot for a precise plot.

In [ ]:
spot = 24252

otm_calls = calls[calls['StrikePrice'] >= spot]
otm_puts  = puts[puts['StrikePrice']  <= spot]

plt.figure(figsize=(11, 6))
plt.plot(otm_puts['StrikePrice'].to_numpy(),  otm_puts['IV'].to_numpy(),  'o-', ms=3, color='indianred', label='OTM Puts')
plt.plot(otm_calls['StrikePrice'].to_numpy(), otm_calls['IV'].to_numpy(), 'o-', ms=3, color='steelblue', label='OTM Calls')
plt.axvline(spot, ls='--', color='gray', alpha=.7, label=f'~Spot ({spot})')
plt.title('NIFTY Volatility Smile — 25-Aug-2026 expiry')
plt.xlabel('Strike'); plt.ylabel('Implied Volatility (%)')
plt.ylim(0, max(otm_puts['IV'].max(), otm_calls['IV'].max()) * 1.15)
plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
def find_iv(mkt, S, K, r, T):
    f = lambda sig: BS.calculateCallOptionPrice(S, K, r, T, sig) - mkt
    try:
        return brentq(f, 0.001, 3.0) * 100
    except ValueError:      # no root in [0.1%, 300%] vol → return NaN instead of crashing
        return np.nan

def find_iv_put(mkt, S, K, r, T):
    # uses the direct BS put (function of sigma) added to Black_Scholes.py
    f = lambda sig: BS.calculatePutOptionPriceBS(S, K, r, T, sig) - mkt
    try:
        return brentq(f, 0.001, 3.0) * 100
    except ValueError:
        return np.nan

In [ ]:
r = 0.065
T = 2 / 365
S = 24252.00

calls['BSIv'] = calls.apply(
    lambda row: find_iv(row['MarketPrice'], S, row['StrikePrice'], r, T),
    axis=1
)
puts['BSIv'] = puts.apply(
    lambda row: find_iv_put(row['MarketPrice'], S, row['StrikePrice'], r, T),
    axis=1
)
calls.head()

In [ ]:
# Reliable IV comes from OUT-OF-THE-MONEY options: puts below spot, calls above.
# Deep-ITM options have near-zero vega, so their inverted IV is numerically unstable.
spot = 24252

otm_puts_bs  = puts[puts['StrikePrice']  <  spot]
otm_calls_bs = calls[calls['StrikePrice'] >= spot]

smile = (pd.concat([otm_puts_bs[['StrikePrice', 'IV', 'BSIv']],
                    otm_calls_bs[['StrikePrice', 'IV', 'BSIv']]])
           .sort_values('StrikePrice').reset_index(drop=True))

plt.figure(figsize=(11, 6))
plt.plot(smile['StrikePrice'].to_numpy(), smile['IV'].to_numpy(),   'o-', ms=3, color='steelblue', label='Exchange IV')
plt.plot(smile['StrikePrice'].to_numpy(), smile['BSIv'].to_numpy(), 'o-', ms=3, color='indianred', label='Black-Scholes IV (inverted)')
plt.axvline(spot, ls='--', color='gray', alpha=.7, label=f'~Spot ({spot})')
plt.title('NIFTY Volatility Smile — Exchange vs Black-Scholes IV (OTM)')
plt.xlabel('Strike'); plt.ylabel('Implied Volatility (%)')
plt.ylim(0, np.nanmax([smile['IV'].max(), smile['BSIv'].max()]) * 1.15)
plt.legend(); plt.grid(alpha=.3); plt.show()